In [1]:
import numpy as np
from bayesian_wrapper import GenerateCrossSection, RunBolsig, ConvertBolsigData, refcrs
from swarmData import swarmData, swarmDatasets
import matplotlib.pyplot as plt
plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Palatino"],
    "font.size": 16,
})

%matplotlib inline

### Test GenerateCrossSection

In [ ]:
theta_ref = np.array([-1.489, 65.0, -82.9, 0.881])
testcrs = GenerateCrossSection(theta_ref)

plt.figure(1)
for c in refcrs.crs:
    if ((c.colType==0) or (c.colType==1)):
        plt.loglog(c.data[:,0],c.data[:,1],'-k')
for c in testcrs.crs:
    if ((c.colType==0) or (c.colType==1)):
        plt.loglog(c.data[:,0],c.data[:,1],'--b')

### Test RunBolsig

In [ ]:
theta_ref = np.array([-1.489, 65.0, -82.9, 0.881])
testcrs = GenerateCrossSection(theta_ref,filename="./bayesian-bolsig/crs/test-crs.txt")

outputs = RunBolsig(dataDir="./bayesian-bolsig")

### Run time measurements

In [ ]:
theta_ref = np.array([-1.489, 65.0, -82.9, 0.881])

from time import perf_counter
times = np.zeros((3,))

Ntest = 10
for k in range(Ntest):
    tic = perf_counter()
    testcrs = GenerateCrossSection(theta_ref,filename="./bayesian-bolsig/crs/test-crs.txt")
    toc = perf_counter()
    times[0] += toc - tic

    outputs, times_bolsig = RunBolsig(dataDir="./bayesian-bolsig",check_time=True)
    times[1:] += times_bolsig
times /= Ntest
print(times)

### Test comparison

In [ ]:
filename = "../swarm/" + swarmDatasets[1] + ".txt"
testData = swarmData(filename)

theta_ref = np.array([-1.489, 65.0, -82.9, 0.881])
testcrs = GenerateCrossSection(theta_ref,filename="./bayesian-bolsig/crs/test-crs.txt")
outputs = RunBolsig(dataDir="./bayesian-bolsig")

bolsigData = ConvertBolsigData(testData.datasets[0],outputs[1])

for k, var in enumerate(testData.datasets[0].variables):
    if ( (var == 'E/N') or (var[-4:] == '-rms') or (var[-4:] == '-max') ): continue
    print(var)
    plt.figure(k)
    plt.errorbar(testData.datasets[0].variables['E/N'],testData.datasets[0].variables[var],yerr=testData.datasets[0].variables[var+'-rms'],fmt='.')
    plt.loglog(testData.datasets[0].variables['E/N'],bolsigData[var],'-r')

## probability evaluation

In [7]:
expDatasets = []
for expDatafile in swarmDatasets:
    filename = "../swarm/" + expDatafile + ".txt"
    expDatasets += [swarmData(filename)]

def log_prior(theta):
    theta_ref = np.array([-1.489, 65.0, -82.9, 0.881])
    sigma2 = ( 0.5 * theta_ref )**2
    return - 0.5 * np.sum( (theta - theta_ref)**2 / sigma2 + np.log(2.0*np.pi*sigma2) )
    
def log_likelihood(theta):
    # compute log-normal probability
    testcrs = GenerateCrossSection(theta,filename="./bayesian-bolsig/crs/test-crs.txt")
    outputs = RunBolsig(dataDir="./bayesian-bolsig")

    lk = 0.0
    for k, expData in enumerate(expDatasets): # per each experiment
        for expDataTable in expData.datasets: # per each table in an experiment
            bolsigData = ConvertBolsigData(expDataTable,outputs[k])
            for kk, var in enumerate(expDataTable.variables): # per each variable in a table
                if ( (var == 'E/N') or (var[-4:] == '-rms') or (var[-4:] == '-max') ): continue
                y = expDataTable.variables[var]
                yerr = expDataTable.variables[var+'-rms']
                sigma2 = np.log(1.0 + yerr / y) ** 2
                pred_y = bolsigData[var]
                lk += - 0.5 * np.sum( (np.log(y) - np.log(pred_y)) ** 2 / sigma2 )
                lk += - 0.5 * np.sum( np.log(2.0*np.pi*sigma2) )

    return lk

def log_posterior(theta):
    lp = log_prior(theta)
    lk = log_likelihood(theta)
    if (not np.isfinite(lp)) or (not np.isfinite(lk)):
        return - np.inf
    return lp + lk

In [16]:
import emcee
nwalkers = 9
theta_ref = np.array([-1.489, 65.0, -82.9, 0.881])
ndim = len(theta_ref)
pos = theta_ref * (1.0 + 0.1 * np.random.randn(nwalkers,ndim) )

sampler = emcee.EnsembleSampler(
    nwalkers, ndim, log_posterior
)
sampler.run_mcmc(pos, 1500, progress=True);

tau = sampler.get_autocorr_time()
print(tau)

100%|████████████████████████████████| 1500/1500 [7:03:23<00:00, 16.94s/it]


AutocorrError: The chain is shorter than 50 times the integrated autocorrelation time for 4 parameter(s). Use this estimate with caution and run a longer chain!
N/50 = 30;
tau: [78.81938357 70.1825076  77.97689902 71.94872444]

In [8]:
# from schwimmbad import MultiPool
import numpy as np
from multiprocessing import Pool, current_process
import multiprocessing

print("Number of cpu : ", multiprocessing.cpu_count())
pool = Pool(8)
np = pool._processes

def func(x):
    from multiprocessing import current_process, cpu_count
    import os
    from time import sleep
    proc = current_process()
    sleep(1.0)
    print(os.getpid() % x, x)
    return x

Number of cpu :  8


### Test writing input file

In [5]:
from input_writer import expConfigs, writeInputFile

filename = './bayesian-bolsig/input/test-input.dat'
crsFile = 'test-crs.txt'
outputfile = 'test-output.dat'

writeInputFile(filename,expConfigs["AlAminLucas1987"],crsFile,outputfile)

NOSCREEN

READCOLLISIONS
"test-crs.txt"
Ar
1

CONDITIONS
VAR
0.0
0.0
300.0
300.0
0.0
0.0
1e+18
1.0
1.0
1
1
1
0.0
200
0
200.0
1e-10
0.0001
1000
1.0
1

RUN
56.5
84.8
113
141
198
254
424
565
678
848
1130
1413
1695
1978
2260
2825
3390
4238
5650

SAVERESULTS
"test-output.dat"
3
1
1
0
0
0
0
0


